Recreation of previous motion-correction processing script using new I/O management approach / refactoring.

[Runtime: ~1min per scan file]

----------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, json, subprocess, shlex, re, glob, math
from datetime import datetime
import pandas as pd
import numpy as np
import nibabel as nib
from typing import Optional
from nilearn.masking import compute_epi_mask
from nilearn import image as nimg
import matplotlib.pyplot as plt
from nilearn.plotting import plot_roi, find_xyz_cut_coords, plot_stat_map
from scipy.ndimage import binary_erosion, gaussian_gradient_magnitude, center_of_mass, gaussian_filter
from skimage.measure import label, regionprops
import nilearn.image as nimg
from nilearn.image import resample_to_img
from nibabel.processing import resample_to_output
import ants

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# --------------------------------------------------------------------
### SET PARAMETERS:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

OVERWRITE_fMRI_MASKS = config['overwrite_EPI_masks']

# First-pass motion-masking parameters:
LOWER_CUTOFF = config['motion_masking_parameters']['lower_cutoff']
UPPER_CUTOFF = config['motion_masking_parameters']['upper_cutoff']
OPENING = config['motion_masking_parameters']['opening']
CONNECTED = config['motion_masking_parameters']['connected']
SMOOTHING_FWHM = config['motion_masking_parameters']['smoothing_FWHM']
RETRY_THRESHOLD = config['motion_masking_parameters']['retry_threshold']

# Fallback masking parameters; only used if first-pass mask coverage exceeds 'retry_threshold':
FALLBACK_LOWER_CUTOFF = config['fallback_parameters']['lower_cutoff']
FALLBACK_UPPER_CUTOFF = config['fallback_parameters']['upper_cutoff']
FALLBACK_OPENING = config['fallback_parameters']['opening']
FALLBACK_CONNECTED = config['fallback_parameters']['connected']

# Motion-correction parameters:
FD_THRESHOLD_MM = config['motion_correction_parameters']['FD_threshold_mm']
RADIUS_MM_FOR_FD = config['motion_correction_parameters']['radius_mm_for_FD']

# ANTS processing parameters:
ANTS_VERBOSE = config['ANTS_parameters']['ants_verbose']
ANTS_MOTION_TRANSFORM = config['ANTS_parameters']['ants_motion_transform']
ANTS_INTERPOLATOR = config['ANTS_parameters']['ants_interpolator']

# --------------------------------------------------------------------
### SET PATHS:

ROOT_DIR = Path(config['root_output_directory'])

### INPUTS:
RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
fMRI_DATA_DIR = config['fMRI_data_directory']
fMRI_PARAMETERS_PATH = Path(ROOT_DIR) / 'fMRI_manifest.csv'

### OUTPUTS:
EPI_REFERENCE_OUTPUT_DIRECTORY = Path(ROOT_DIR) / config['EPI_ref_dir']
MASK_OUTPUT_DIRECTORY = Path(ROOT_DIR) / config['motion_mask_dir']
MOTION_XFORMS_SUBDIR = Path(ROOT_DIR) / config['motion_xforms_dir']
CONFOUNDS_SUBDIR = Path(ROOT_DIR) / config['confounds_dir']
OUT_MOTION_DIR = Path(ROOT_DIR) / config['motion_correction_output_dir']

QC_OUTPUT_DIR = Path(ROOT_DIR) / config['motion_correction_QC_subdir']

# --------------------------------------------------------------------
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs = pd.read_csv(fMRI_PARAMETERS_PATH)

In [ ]:
# =====================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# =====================================================================

# ---- FILTERING (if enabled) ----
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to fMRI_runs...")
    print(f"[FILTER] Starting with {len(fMRI_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(fMRI_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = fMRI_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(fMRI_runs):,} rows.\n")

# ---- DIAGNOSTIC SUBSETTING (if enabled) ----
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    fMRI_runs = fMRI_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    fMRI_runs = fMRI_runs.reset_index(drop=True)
    display(fMRI_runs)

FIRST STEP: Perform "data audit" to validate output from the previous pipeline stage (MRI reconstructions via FreeSurfer):

In [ ]:
# =========================
# FreeSurfer prerequisite audit driven by pre-built fMRI_runs manifest
# =========================

# Use SUBJECTS_DIR:
SUBJECTS_DIR = str(FREESURFER_SUBJECTS_DIR)

# Build target_subject_list (order-preserving unique from fMRI_runs):
target_subject_list = []
_seen = set()
for _sid in fMRI_runs['subject_ID'].astype(str):
    if _sid not in _seen:
        target_subject_list.append(_sid)
        _seen.add(_sid)

# Strict check per your note: nii_filetype must be populated (error if not)
if fMRI_runs['nii_filetype'].isna().any() or (fMRI_runs['nii_filetype'].astype(str).str.strip() == '').any():
    raise RuntimeError("Found missing/empty entries in 'nii_filetype'; upstream manifest should have dropped these rows.")

### Perform specific file spot-checks for core anatomical data within each subject's FreeSurfer directory (and folders within):
CORE_FILES = {
    "mri/T1.mgz":            "has_T1_mgz",
    "mri/brain.mgz":         "has_brain_mgz",
    "mri/brainmask.mgz":     "has_brainmask_mgz",
    "mri/aseg.mgz":          "has_aseg_mgz"}
NICE_TO_HAVE = {
    "mri/wm.mgz":            "has_wm_mgz",
    "mri/wmparc.mgz":        "has_wmparc_mgz",
    "mri/aparc+aseg.mgz":    "has_aparc_aseg_mgz"}
SURF_FILES = {
    "surf/lh.white":         "has_lh_white",
    "surf/rh.white":         "has_rh_white",
    "surf/lh.pial":          "has_lh_pial",
    "surf/rh.pial":          "has_rh_pial"}
STATUS_FILES = {
    "scripts/recon-all.done":   "has_recon_done",
    "scripts/recon-all.status": "has_recon_status"}

ALL_SPEC = {}
ALL_SPEC.update(CORE_FILES)
ALL_SPEC.update(NICE_TO_HAVE)
ALL_SPEC.update(SURF_FILES)
ALL_SPEC.update(STATUS_FILES)

def audit_fs_subject(subject_ID, subjects_dir=SUBJECTS_DIR):
    subj_dir = os.path.join(subjects_dir, subject_ID)
    out = {
        "subject_ID": subject_ID,
        "fs_subject_dir": subj_dir,
        "fs_dir_exists": os.path.isdir(subj_dir)}
    for rel, col in ALL_SPEC.items():
        out[col] = False

    if not out["fs_dir_exists"]:
        out["status"] = "BLOCK"
        out["reason"] = "FreeSurfer subject dir missing"
        return out

    missing_core = []
    for rel, col in ALL_SPEC.items():
        p = os.path.join(subj_dir, rel)
        present = os.path.exists(p)
        out[col] = bool(present)
        if (rel in CORE_FILES) and (not present):
            missing_core.append(rel)

    if missing_core:
        out["status"] = "BLOCK"
        out["reason"] = "missing core: " + ", ".join(missing_core[:6]) + (" ..." if len(missing_core) > 6 else "")
    else:
        surf_ok = out["has_lh_white"] and out["has_rh_white"] and out["has_lh_pial"] and out["has_rh_pial"]
        wm_ok   = out["has_wm_mgz"] or out["has_wmparc_mgz"] or out["has_aparc_aseg_mgz"]
        hints = []
        if not surf_ok: hints.append("no_surfaces")
        if not wm_ok:   hints.append("no_wmseg_like")
        out["status"] = "OK" if not hints else "OK_WITH_NOTES"
        out["reason"] = "; ".join(hints) if hints else ""
    return out

# Run audit on subjects from fMRI_runs:
_rows = [audit_fs_subject(subject_ID, subjects_dir=SUBJECTS_DIR) for subject_ID in target_subject_list]
anatomical_checks = pd.DataFrame.from_records(_rows).sort_values(["subject_ID"])

# Console summary:
_counts = anatomical_checks["status"].value_counts(dropna=False)
print("\n[INFO] Anatomy status counts (targets):")
print(_counts.to_string())

# Handle blocking issues according to HARD_STOP:
if "BLOCK" in _counts.index and _counts["BLOCK"] > 0:
    _blocked = anatomical_checks[anatomical_checks["status"] == "BLOCK"]
    print("\n[ERROR] Target subjects with blocking issues (first 10):")
    _show_cols = ["subject_ID","reason","fs_subject_dir"]
    print(_blocked[_show_cols].head(10).to_string(index=False))

    if HARD_STOP:
        raise RuntimeError(f"{_counts['BLOCK']} target subject(s) are missing core anatomy prerequisites.")
    else:
        _blocked_sids = _blocked["subject_ID"].astype(str).tolist()
        _before = len(fMRI_runs)
        fMRI_runs = fMRI_runs[~fMRI_runs["subject_ID"].astype(str).isin(_blocked_sids)].copy()
        _after = len(fMRI_runs)
        print(f"\n[warn] Dropped {_before - _after} fMRI row(s) across {len(_blocked_sids)} subject(s) due to missing FS prerequisites.")
        # Recompute counts for remaining cohort (optional informational print):
        _remaining_sids = sorted(set(fMRI_runs['subject_ID'].astype(str)))
        print(f"[info] Remaining subjects after drop: {len(_remaining_sids)}")

# Non-blocking notes:
_notes = anatomical_checks[anatomical_checks["status"] == "OK_WITH_NOTES"]
if not _notes.empty:
    print("\n[INFO] Target subjects with non-blocking notes (first 10):")
    print(_notes[["subject_ID","reason"]].head(10).to_string(index=False))

if not ("BLOCK" in _counts.index and _counts["BLOCK"] > 0 and HARD_STOP):
    print("\n[OK] Anatomy prerequisites check completed (blocking issues handled per HARD_STOP).")


-------------
**NEXT STEP: Build EPI references and motion masks:**


First we define the function for building masks and EPI references; then we iterate over 'target_subject_list', and retrieve .nii filepaths and other core variables & parameters for each target subject_ID, then submitting all of this into the main function which will save the two outputs into the chosen subdirectories within the main output folder.

The **median mask** is just taking the median intensity value of every voxel scanned; this yields one "crisp and clean" 3d image of the anatomical volume; the **EPI reference** is built from this initial mask, and creates a 3d map designating which voxels are brain and which are not-brain (i.e. it's fully binary; every voxel is labeled as 1/0 depending on if it has been segmented as part of the brain, vs. as any other tissue type).

In [ ]:
# =========================
# Stage A — Build per-run EPI reference & mask (site-agnostic)
# =========================

"""
- Drops dummy volumes, computes temporal-median EPI reference
- Builds Nilearn EPI mask with configured thresholds (+ stricter fallback if coverage too high)
- Returns per-run QC (mask coverage, median intensity, volumes before/after, TR)
- Does NOT perform motion estimation here -- that happens in Stage B
"""

def build_epi_reference_and_mask(
    *,
    nii_path: str,
    drop_first_n: int,
    ref_out_dir: str,
    mask_out_dir: str,
    ref_name: str,
    mask_name: str,
    nilearn_kwargs: Optional[dict] = None,
    overwrite: bool = False):

    # Create a temporal median EPI reference (after dropping dummy frames) and a Nilearn EPI brain mask:
    os.makedirs(ref_out_dir, exist_ok=True)
    os.makedirs(mask_out_dir, exist_ok=True)

    ref_path  = os.path.join(ref_out_dir,  ref_name)
    mask_path = os.path.join(mask_out_dir, mask_name)

    # Fast skip if both already exist (and overwrite=False) — but still compute QC metrics from disk:
    if (not overwrite) and os.path.exists(ref_path) and os.path.exists(mask_path):
        # Header / geometry from original 4D (for n_vols + TR + pixdim)
        img4d = nib.load(nii_path)
        nx, ny, nz, nt = img4d.shape
        pixdim = img4d.header.get_zooms()[:3]
        TR_hdr = float(img4d.header.get_zooms()[3]) if len(img4d.header.get_zooms()) >= 4 else np.nan

        # Compute mask QC metrics using the *saved* ref + mask:
        ref_img  = nib.load(ref_path)
        mask_img = nib.load(mask_path)

        # Basic lattice sanity-check (warn only; do not hard-fail):
        if ref_img.shape != mask_img.shape or not np.allclose(ref_img.affine, mask_img.affine, atol=1e-5):
            print(f"[WARN] Existing ref/mask lattice mismatch for: {os.path.basename(ref_path)}")

        ref_data  = ref_img.get_fdata(dtype=np.float32)
        mask_data = mask_img.get_fdata(dtype=np.float32)

        # Robust binarization (handle 3D/4D masks):
        if mask_data.ndim == 4:
            mask_data = mask_data[..., 0]
        mask_bin = (mask_data > 0.5)

        total_voxels = int(np.prod(ref_img.shape[:3]))
        mask_voxels  = int(np.count_nonzero(mask_bin))
        mask_frac    = (mask_voxels / total_voxels) if total_voxels > 0 else np.nan
        med_intensity = float(np.median(ref_data[mask_bin])) if mask_voxels > 0 else np.nan

        return {
            "ref_path": ref_path,
            "mask_path": mask_path,
            "skipped": True,
            "reason": "exists",
            "n_vols_before": int(nt),
            "drop_first_n": int(drop_first_n),
            "n_vols_after": max(0, int(nt) - int(drop_first_n)),
            "TR_hdr": TR_hdr,
            "vox_x": float(pixdim[0]),
            "vox_y": float(pixdim[1]),
            "vox_z": float(pixdim[2]),
            "mask_voxels": mask_voxels,
            "mask_fraction": mask_frac,
            "ref_median_intensity_in_mask": med_intensity,
            "mask_retried_due_to_high_coverage": False}

    # Load 4D image; drop dummy frames; compute median:
    img = nib.load(nii_path)
    data = img.get_fdata(dtype=np.float32)
    if data.ndim != 4:
        raise ValueError(f"{nii_path} is not 4D (ndim={data.ndim}).")
    nx, ny, nz, nt = data.shape
    dummy_frames = int(drop_first_n)
    if nt <= dummy_frames:
        raise ValueError(f"Not enough volumes after dropping {dummy_frames} (nt={nt}).")

    data_steady = data[..., dummy_frames:]
    ref_vol = np.median(data_steady, axis=3).astype(np.float32)
    ref_img = nib.Nifti1Image(ref_vol, img.affine, header=img.header)
    ref_img.set_data_dtype(np.float32)
    nib.save(ref_img, ref_path)

    # Generate brain mask via nilearn (with optional smoothing & thresholding):
    nk = dict(nilearn_kwargs or {})
    smoothing_fwhm = nk.pop("smoothing_fwhm", None)
    ref_for_mask = nimg.smooth_img(ref_img, smoothing_fwhm) if smoothing_fwhm else ref_img

    mask_img = compute_epi_mask(ref_for_mask, **nk)

    # QC on mask (uses stricter fallback parameters if coverage % appears over-inclusive):
    mask_data = mask_img.get_fdata(dtype=np.float32)
    mask_bin = (mask_data > 0.5)
    total_voxels = int(nx * ny * nz)
    mask_voxels = int(np.count_nonzero(mask_bin))
    mask_frac = mask_voxels / total_voxels if total_voxels > 0 else np.nan
    med_intensity = float(np.median(ref_vol[mask_bin])) if mask_voxels > 0 else np.nan

    retried = False
    if np.isfinite(mask_frac) and (mask_frac > RETRY_THRESHOLD):
        retried = True
        #####################################################################################
        # Stricter, site-agnostic fallback parameters:
        strict_kw = {"lower_cutoff": FALLBACK_LOWER_CUTOFF,
                     "upper_cutoff": FALLBACK_UPPER_CUTOFF,
                     "opening": FALLBACK_OPENING,
                     "connected": FALLBACK_CONNECTED}
        #####################################################################################
        mask_img = compute_epi_mask(ref_img, **strict_kw)  # <-- uses unsmoothed reference for fallback branch
        mask_data = mask_img.get_fdata(dtype=np.float32)
        mask_bin = (mask_data > 0.5)
        mask_voxels = int(np.count_nonzero(mask_bin))
        mask_frac = mask_voxels / total_voxels if total_voxels > 0 else np.nan
        med_intensity = float(np.median(ref_vol[mask_bin])) if mask_voxels > 0 else np.nan
        print(f"[INFO] Recomputed mask with stricter thresholds; coverage={mask_frac:.2f}")

    nib.save(mask_img, mask_path)

    TR_hdr = float(img.header.get_zooms()[3]) if len(img.header.get_zooms()) >= 4 else np.nan
    pixdim = img.header.get_zooms()[:3]

    return {
        "ref_path": ref_path,
        "mask_path": mask_path,
        "skipped": False,
        "n_vols_before": int(nt),
        "drop_first_n": dummy_frames,
        "n_vols_after": int(nt - dummy_frames),
        "TR_hdr": TR_hdr,
        "vox_x": float(pixdim[0]), "vox_y": float(pixdim[1]), "vox_z": float(pixdim[2]),
        "mask_voxels": mask_voxels,
        "mask_fraction": mask_frac,
        "ref_median_intensity_in_mask": med_intensity,
        "mask_retried_due_to_high_coverage": retried}

# -------------------------
# Build EPI references + brain masks:
# -------------------------

if fMRI_runs['nii_filetype'].isna().any() or (fMRI_runs['nii_filetype'].astype(str).str.strip() == '').any():
    raise RuntimeError("Found missing/empty entries in 'nii_filetype'; upstream manifest should have dropped these rows.")

qc_rows = []

for _, r in fMRI_runs.iterrows():
    subject_ID   = str(r['subject_ID'])
    condition_ID = str(r['group_ID'])
    session_ID   = str(r['session_ID'])

    # Materialize full path from basepath + filetype (note: already includes leading '.'):
    nii_path = str(Path(str(r['fMRI_basepath']) + str(r['nii_filetype'])))

    # Pull per-run parameters from manifest:
    n_vols_before = int(r['n_t'])
    n_dummy_vols  = int(r['drop_first_n'])
    n_vox_x       = int(r['n_x']); n_vox_y = int(r['n_y']); n_vox_z = int(r['n_z'])
    TR_param      = float(r['RepetitionTime'])

    # Set output artifact names:
    prefix   = f"{subject_ID}_{session_ID}"
    ref_name = f"{prefix}_boldref.nii"
    msk_name = f"{prefix}_boldref_mask.nii"

    # Set nilearn mask keyword args from config parameters:
    mask_kwargs = {
        "lower_cutoff": LOWER_CUTOFF,
        "upper_cutoff": UPPER_CUTOFF,
        "opening": OPENING,
        "connected": CONNECTED,
        "smoothing_fwhm": SMOOTHING_FWHM}

    qc = build_epi_reference_and_mask(
        nii_path=nii_path,
        drop_first_n=n_dummy_vols,
        ref_out_dir=str(EPI_REFERENCE_OUTPUT_DIRECTORY),
        mask_out_dir=str(MASK_OUTPUT_DIRECTORY),
        ref_name=ref_name,
        mask_name=msk_name,
        nilearn_kwargs=mask_kwargs,
        overwrite=OVERWRITE_fMRI_MASKS)

    # Preserve original QC enrichment fields:
    qc.update({
        "subject_ID": subject_ID,
        "condition_ID": condition_ID,
        "session_ID": session_ID,
        "nii_path": nii_path,
        "TR_param": TR_param,
        "n_vols_param": n_vols_before,
        "n_x": n_vox_x, "n_y": n_vox_y, "n_z": n_vox_z})

    # Consistency warnings:
    if qc["n_vols_before"] != n_vols_before:
        print(f"[WARN] [{subject_ID}] Header nt={qc['n_vols_before']} != manifest n_t={n_vols_before}")
    if not np.isnan(qc["TR_hdr"]) and abs(qc["TR_hdr"] - TR_param) > 1e-3:
        print(f"[WARN] [{subject_ID}] Header TR={qc['TR_hdr']:.4f} differs from TR_param={TR_param:.4f}")

    qc_rows.append(qc)

qc_df = pd.DataFrame.from_records(qc_rows)
qc_df["n_vols_after_ok"] = qc_df["n_vols_after"] >= 120

qc_df.head()

Print summary log of files that were re-processed using the fallback motion-masking parameters:

In [ ]:
qc_df.columns

In [ ]:
qc_df[qc_df['mask_retried_due_to_high_coverage'] == True][['subject_ID', 'condition_ID', 'session_ID', 'mask_fraction']]

In [ ]:
# =========================
# Stage B — Motion estimation
# =========================

"""
Stage B — Motion estimation ONLY (no resampling, yet)

- Uses median EPI reference + robust mask (outputs from Stage A, previous cell)
- Calls ants.motion_correction with type_of_transform=ANTS_MOTION_TRANSFORM ("BOLDRigid" by default), mask, and fdOffset=RADIUS_MM_FOR_FD
- Saves ANTs transforms in subject-specific subdirectories (under '.../transforms/<prefix>/')
- Writes a BIDS(-ish) 'confounds.tsv' file containing framewise_displacement (ANTs FD, post-dummy-drop) & DVARS measures
- Produces a console summary per run
"""

def _dvars_from_nii(nii_path:str, mask_path:str, drop_first_n:int) -> np.ndarray:
    # DVARS on masked 4D after dropping dummies (first kept frame gets 0.0):
    img = nib.load(nii_path)
    data = img.get_fdata(dtype=np.float32)
    if data.ndim != 4:
        raise ValueError(f"{nii_path} is not 4D")
    T = data.shape[3]
    if drop_first_n >= T-1:
        raise ValueError(f"drop_first_n={drop_first_n} leaves <2 frames in {os.path.basename(nii_path)}")

    mask = nib.load(mask_path).get_fdata().astype(bool)
    if mask.ndim == 4:
        mask = mask[..., 0]
    if mask.sum() == 0:
        raise ValueError(f"Mask has zero voxels: {mask_path}")

    dat = data[..., drop_first_n:T]
    Tkept = dat.shape[3]
    vox_ts = dat[mask].reshape(-1, Tkept)
    d = np.diff(vox_ts, axis=1)
    dvars = np.sqrt((d**2).mean(axis=0))
    return np.concatenate([np.array([0.0], dtype=np.float32), dvars.astype(np.float32)])

def run_motion_for_run(*, subject_ID:str, condition_ID:str, session_ID:str,
                       nii_path:str, ref_path:str, mask_path:str,
                       drop_first_n:int, TR_s:float):
    """
    Estimates per-frame rigid motion vs. provided EPI reference using ANTsPyX; saves
    transforms to disk, and writes a 'confounds.tsv' file (FD from ANTs, DVARS):
    """
    prefix = f"{subject_ID}_{session_ID}"

    # Paths (adapted to new configured directories)
    xform_dir = Path(MOTION_XFORMS_SUBDIR) / prefix
    xform_dir.mkdir(parents=True, exist_ok=True)
    outprefix = str(xform_dir / "mc")  # ANTs will append per-frame suffixes

    conf_dir = Path(CONFOUNDS_SUBDIR)
    conf_dir.mkdir(parents=True, exist_ok=True)
    conf_tsv = str(conf_dir / f"{prefix}_confounds.tsv")
    conf_json = str(conf_dir / f"{prefix}_confounds.json")

    # Sanity
    if not os.path.exists(nii_path):  raise FileNotFoundError(f"4D fMRI not found: {nii_path}")
    if not os.path.exists(ref_path):  raise FileNotFoundError(f"EPI reference not found: {ref_path}")
    if not os.path.exists(mask_path): raise FileNotFoundError(f"EPI mask not found: {mask_path}")

    img4d = ants.image_read(nii_path)   # <-- 4D
    fixed = ants.image_read(ref_path)   # <-- 3D
    mask  = ants.image_read(mask_path)  # <-- 3D

    # Motion estimation with ANTs-provided Framewise Displacement (uses head radius global parameter):
    mc = ants.motion_correction(
        img4d,
        fixed=fixed,
        type_of_transform=ANTS_MOTION_TRANSFORM,
        mask=mask,
        fdOffset=RADIUS_MM_FOR_FD,
        outprefix=outprefix,
        verbose=ANTS_VERBOSE)

    # FD from ANTs (aligned to original T):
    FD = np.asarray(mc["FD"], dtype=np.float32)
    FD_kept = FD[drop_first_n:].copy()
    if FD_kept.size == 0:
        raise ValueError(f"After dropping {drop_first_n} dummies, no frames remain for {prefix}")
    FD_kept[0] = 0.0  # <-- first kept frame has no previous neighbor

    # DVARS (on original 4D data, after dummy-frame-drop):
    DVARS = _dvars_from_nii(nii_path, mask_path, drop_first_n)
    n_kept = FD_kept.shape[0]
    if DVARS.shape[0] != n_kept:
        raise RuntimeError(f"Length mismatch FD({n_kept}) vs DVARS({DVARS.shape[0]}) for {prefix}")

    # Build confounds TSV:
    conf = pd.DataFrame({
        "frame": np.arange(n_kept, dtype=int),
        "framewise_displacement": FD_kept,
        "dvars": DVARS})
    conf["motion_outlier"] = (conf["framewise_displacement"] > FD_THRESHOLD_MM).astype(int)
    conf["n_dummy_dropped"] = int(drop_first_n)
    conf["tr_s"] = float(TR_s)
    conf["radius_mm_for_FD"] = float(RADIUS_MM_FOR_FD)
    conf.to_csv(conf_tsv, sep="\t", index=False)

    # Write JSON provenance file:
    meta = {
        "Description": "Rigid motion estimation vs. EPI reference; no 4D resampling performed.",
        "ANTsPy_function": "ants.motion_correction",
        "type_of_transform": ANTS_MOTION_TRANSFORM,
        "fdOffset_radius_mm": RADIUS_MM_FOR_FD,
        "FD_threshold_mm": FD_THRESHOLD_MM,
        "inputs": {
            "bold_4d": os.path.abspath(nii_path),
            "epi_reference": os.path.abspath(ref_path),
            "epi_mask": os.path.abspath(mask_path),
            "drop_first_n": int(drop_first_n),
            "TR_s": float(TR_s)},
        "outputs": {
            "transforms_dir": os.path.abspath(str(xform_dir)),
            "confounds_tsv": os.path.abspath(conf_tsv)}}
    with open(conf_json, "w") as f:
        json.dump(meta, f, indent=2)

    # Console summary:
    msg = (f"\t[MOTION] {prefix}: T={img4d.shape[3]}  drop={drop_first_n}  keep={n_kept}  "
           f"FD(mean/p95/max)={FD_kept.mean():.3f}/{np.percentile(FD_kept,95):.3f}/{FD_kept.max():.3f}  "
           f"censored={int((conf['motion_outlier']==1).sum())}")
    print(msg)

    return {
        "prefix": prefix,
        "n_frames_original": int(img4d.shape[3]),
        "n_dummy_dropped": int(drop_first_n),
        "n_frames_kept": int(n_kept),
        "fd_mean": float(FD_kept.mean()),
        "fd_p95": float(np.percentile(FD_kept, 95)),
        "fd_max": float(FD_kept.max()),
        "n_censored": int((conf["motion_outlier"] == 1).sum()),
        "confounds_tsv": conf_tsv,
        "transforms_dir": str(xform_dir)}

# ______________________________________________________________________________________________
# Main loop execution:
summaries = []

for i, r in enumerate(fMRI_runs.itertuples(index=False), start=1):
    subject_ID   = str(r.subject_ID)
    condition_ID = str(r.group_ID)
    session_ID   = str(r.session_ID)

    print(f"Processing subject_ID: {subject_ID} [{i}/{len(fMRI_runs)}]...")

    # Materialize paths/numbers from manifest:
    nii_path     = str(Path(str(r.fMRI_basepath) + str(r.nii_filetype)))  # note: nii_filetype already includes leading '.'
    drop_first_n = int(r.drop_first_n)
    TR_s         = float(r.RepetitionTime)

    # Stage A outputs:
    prefix   = f"{subject_ID}_{session_ID}"
    ref_path  = str(Path(EPI_REFERENCE_OUTPUT_DIRECTORY) / f"{prefix}_boldref.nii")
    mask_path = str(Path(MASK_OUTPUT_DIRECTORY)          / f"{prefix}_boldref_mask.nii")

    summaries.append(
        run_motion_for_run(
            subject_ID=subject_ID,
            condition_ID=condition_ID,
            session_ID=session_ID,
            nii_path=nii_path,
            ref_path=ref_path,
            mask_path=mask_path,
            drop_first_n=drop_first_n,
            TR_s=TR_s))

summary_df = pd.DataFrame(summaries)
summary_df

In [ ]:
# =========================
# Stage C — Compute & compile motion-corrected 4D volume
# =========================

def apply_motion_transforms_for_run(*, subject_ID, condition_ID, session_ID, nii_path):
    """
    Apply per-frame ANTs affines to produce motion-corrected 4D on the BOLDREF lattice (subject-/machine-native EPI space).
    Critically: copy full moving-image geometry (origin/direction/spacing) onto each frame before applying the transform.
    """
    prefix    = f"{subject_ID}_{session_ID}"
    xform_dir = Path(MOTION_XFORMS_SUBDIR) / prefix
    ref_path  = Path(EPI_REFERENCE_OUTPUT_DIRECTORY) / f"{prefix}_boldref.nii"   # distorted BOLDREF used as fixed
    out_mc    = Path(OUT_MOTION_DIR) / f"{prefix}_bold_mc.nii"

    # Check prerequisites:
    if not xform_dir.exists():
        raise FileNotFoundError(f"Transforms dir not found: {xform_dir}")
    if not ref_path.exists():
        raise FileNotFoundError(f"Reference not found: {ref_path}")
    if not Path(nii_path).exists():
        raise FileNotFoundError(f"BOLD 4D not found: {nii_path}")

    # Per-frame transforms (in filename order):
    tfms = sorted(glob.glob(str(xform_dir / "mc*GenericAffine.mat")))

    # Load fixed & moving-geometry template:
    fixed    = ants.image_read(str(ref_path))   # <-- fixed lattice (BOLDREF)
    mov_geom = ants.image_read(str(ref_path))   # <-- use BOLDREF as geometry template for each moving frame

    # Load 4D data:
    img  = nib.load(nii_path)
    data = img.get_fdata(dtype=np.float32)
    if data.ndim != 4:
        raise ValueError(f"{nii_path} is not 4D")
    X, Y, Z, T = data.shape

    # Sanity-check transform count vs n_frames:
    if len(tfms) != T:
        print(f"[WARN] {prefix}: #transforms={len(tfms)} != #frames={T}. Proceeding with min().")
    n_apply = min(T, len(tfms))

    mc_np = np.zeros_like(data, dtype=np.float32)

    # Frame-by-frame apply:
    for t in range(n_apply):
        mov_t = ants.from_numpy(data[..., t].astype(np.float32))
        mov_t = ants.copy_image_info(mov_geom, mov_t)  # <<< keep CRITICAL fix

        warped = ants.apply_transforms(
            fixed=fixed,
            moving=mov_t,
            transformlist=[tfms[t]],
            interpolator=ANTS_INTERPOLATOR)

        mc_np[..., t] = warped.numpy().astype(np.float32)

        if (t + 1) % max(1, T // 10) == 0 or t == T - 1:
            print(f"    frame {t+1}/{T} done")

    # For any trailing frames without transforms, copy originals (e.g., if transforms are fewer than frames):
    if n_apply < T:
        mc_np[..., n_apply:] = data[..., n_apply:]

    # Preserve TR and spatial zooms (from fixed/ref):
    fixed_nib   = nib.load(str(ref_path))
    spacing_xyz = tuple(float(s) for s in fixed_nib.header.get_zooms()[:3])
    zooms_in    = img.header.get_zooms()
    TR          = float(zooms_in[3]) if len(zooms_in) >= 4 else 0.0

    out_img = nib.Nifti1Image(mc_np, fixed_nib.affine, header=fixed_nib.header)
    out_img.header.set_xyzt_units('mm', 'sec')
    out_img.header.set_zooms(spacing_xyz + (TR,))
    out_mc.parent.mkdir(parents=True, exist_ok=True)
    nib.save(out_img, str(out_mc))

    print(f"[APPLY] Wrote motion-corrected 4D: {out_mc}")

    # -------------------------------
    # Minimal QC check for nonzero support in GM∧Atlas on EPI lattice
    # NOTE: These data objects normally do NOT exist yet; this check will only trigger
    #       on a second pass through the pipeline when alignment/bbregister have already been run.
    # -------------------------------
    try:
        atlas_fp = Path(ROOT_DIR) / "alignment"  / prefix / "atlas_craddock_on_EPI.nii.gz"
        gm_fp    = Path(ROOT_DIR) / "bbregister" / prefix / "gm_epi.nii.gz"

        # NEW conditional guard: only run the QC if both files already exist:
        if atlas_fp.exists() and gm_fp.exists():
            mc    = nib.load(str(out_mc))
            atlas = nib.load(str(atlas_fp))
            gm    = nib.load(str(gm_fp))

            same_shape         = (mc.shape[:3] == atlas.shape[:3] == gm.shape[:3])
            same_aff_mc_atlas  = np.allclose(mc.affine, atlas.affine, atol=1e-4)
            same_aff_mc_gm     = np.allclose(mc.affine, gm.affine,    atol=1e-4)
            if not (same_shape and same_aff_mc_atlas and same_aff_mc_gm):
                print(f"[MC QC WARN] {prefix}: lattice mismatch (shape/affine). QC may be unreliable.")

            mc_np_vol = mc.get_fdata(dtype=np.float32)
            atlas_i   = atlas.get_fdata().astype(np.int32)
            gm_m      = gm.get_fdata() > 0.5

            roi_mask = (atlas_i > 0) & gm_m
            if roi_mask.sum() == 0:
                print(f"[MC QC WARN] {prefix}: GM∧Atlas has 0 voxels.")
            else:
                nz = (np.max(np.abs(mc_np_vol), axis=3) > 0)
                support = float(nz[roi_mask].mean())
                print(f"[MC QC] {prefix}: GM∧Atlas nonzero support = {support:.3f}")

        # else-branch is now silent (no [MC QC SKIP] message), but can be restored if desired:
        # else:
        #     print(f"[MC QC SKIP] {prefix}: missing GM/Atlas QC inputs.")

    except Exception as e:
        print(f"[MC QC ERR] {prefix}: {e}")

    return str(out_mc)

# _______________________________________________________________________
# Main loop execution:
applied = []
for i, r in enumerate(fMRI_runs.itertuples(index=False), start=1):
    subject_ID   = str(r.subject_ID)
    condition_ID = str(r.group_ID)
    session_ID   = str(r.session_ID)

    # Construct input BOLD path from manifest (note that 'nii_filetype' already includes leading '.'):
    nii_path = str(Path(str(r.fMRI_basepath) + str(r.nii_filetype)))

    mc_path = apply_motion_transforms_for_run(
        subject_ID=subject_ID,
        condition_ID=condition_ID,
        session_ID=session_ID,
        nii_path=nii_path)
    applied.append({"subject_ID": subject_ID, "session_ID": session_ID, "mc_path": mc_path})

pd.DataFrame(applied)

Final step; QC checks:

In [ ]:
# =========================
# Stage D — Centralized QC
# =========================

# QC directory (use config key if present; otherwise default):
QC_dir = QC_OUTPUT_DIR
QC_dir.mkdir(parents=True, exist_ok=True)
QC_CSV = QC_dir / "motion_qc_summary.csv"

def qc_plot_fd_dvars(conf_tsv, out_png, TR_s):
    df = pd.read_csv(conf_tsv, sep="\t")
    t = np.arange(len(df)) * TR_s
    fig, ax1 = plt.subplots(figsize=(9, 4))
    ax1.plot(t, df["framewise_displacement"].values, label="FD (mm)")
    ax1.axhline(FD_THRESHOLD_MM, linestyle="--", linewidth=1, label=f"FD thr {FD_THRESHOLD_MM} mm")
    ax1.set_xlabel("Time (s)"); ax1.set_ylabel("FD (mm)")
    ax2 = ax1.twinx()
    ax2.plot(t, df["dvars"].values, alpha=0.5, label="DVARS")
    ax2.set_ylabel("DVARS (a.u.)")
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc="upper right")
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_png, dpi=150)
    plt.close(fig)

def qc_mask_overlay(ref_path, mask_path, out_png, *, atol_affine=1e-5):
    ref_img = nib.load(ref_path)
    mask_img = nib.load(mask_path)
    if ref_img.shape != mask_img.shape:
        raise RuntimeError(f"Shape mismatch: ref {ref_img.shape} vs mask {mask_img.shape}")
    if not np.allclose(ref_img.affine, mask_img.affine, atol=atol_affine):
        raise RuntimeError("Affine mismatch between ref and mask")

    ref_data = ref_img.get_fdata()
    msk_data = mask_img.get_fdata() > 0.5
    mask_frac = float(msk_data.mean())
    med_in_mask = float(np.median(ref_data[msk_data])) if msk_data.any() else np.nan

    try:
        cut_coords = tuple(find_xyz_cut_coords(mask_img).tolist())
    except Exception:
        try:
            cut_coords = tuple(find_xyz_cut_coords(ref_img).tolist())
        except Exception:
            cut_coords = None

    display = plot_roi(mask_img, bg_img=ref_img, display_mode="ortho",
                       cut_coords=cut_coords,
                       title=f"Mask overlay | coverage={mask_frac:.2f} | med_int={med_in_mask:.1f}")
    display.savefig(out_png, dpi=150)
    display.close()
    return mask_frac, med_in_mask

# _____________________________________________________________________________________
# Main loop execution:
qc_rows = []
for i, r in enumerate(fMRI_runs.itertuples(index=False), start=1):
    subject_ID   = str(r.subject_ID)
    condition_ID = str(r.group_ID)
    session_ID   = str(r.session_ID)
    TR_s         = float(r.RepetitionTime)

    prefix   = f"{subject_ID}_{session_ID}"
    ref_path = Path(EPI_REFERENCE_OUTPUT_DIRECTORY) / f"{prefix}_boldref.nii"
    mask_path = Path(MASK_OUTPUT_DIRECTORY) / f"{prefix}_boldref_mask.nii"
    conf_path = Path(CONFOUNDS_SUBDIR) / f"{prefix}_confounds.tsv"

    fd_png   = QC_dir / f"{prefix}_01_fd_dvars.png"
    mask_png = QC_dir / f"{prefix}_02_mask_overlay.png"

    if not (ref_path.exists() and mask_path.exists() and conf_path.exists()):
        print(f"[WARN] Missing one of ref/mask/confounds for {prefix}; skipping QC")
        continue

    # FD/DVARS plot:
    qc_plot_fd_dvars(str(conf_path), str(fd_png), TR_s)

    # Brain mask overlay (+ metrics):
    try:
        mask_frac, med_int = qc_mask_overlay(str(ref_path), str(mask_path), str(mask_png))
    except Exception as e:
        print(f"[ERROR] {prefix}: {e} — skipping overlay + metrics")
        continue

    dfc = pd.read_csv(conf_path, sep="\t")
    fd_mean = dfc["framewise_displacement"].mean()
    fd_p95  = np.percentile(dfc["framewise_displacement"], 95)
    fd_max  = dfc["framewise_displacement"].max()
    n_cens  = int((dfc["motion_outlier"] == 1).sum())
    n_kept  = len(dfc)

    qc_rows.append({
        "subject_ID": subject_ID,
        "condition_ID": condition_ID,
        "session_ID": session_ID,
        "fd_mean": fd_mean, "fd_p95": fd_p95, "fd_max": fd_max,
        "n_censored": n_cens, "n_frames_kept": n_kept,
        "mask_fraction": mask_frac, "ref_median_intensity_in_mask": med_int,
        "fd_plot": str(fd_png), "mask_overlay": str(mask_png)})

# Append-safe CSV (de-duplicates on subject/session):
if qc_rows:
    new_df = pd.DataFrame(qc_rows)
    if QC_CSV.exists():
        old = pd.read_csv(QC_CSV)
        if all(c in old.columns for c in ["subject_ID","session_ID"]):
            existing_keys = set(old["subject_ID"].astype(str) + "|" + old["session_ID"].astype(str))
            new_keys = new_df["subject_ID"].astype(str) + "|" + new_df["session_ID"].astype(str)
            old = old[~(old["subject_ID"].astype(str).add("|").add(old["session_ID"].astype(str)).isin(set(new_keys)))]
        out_df = pd.concat([old, new_df], ignore_index=True)
    else:
        out_df = new_df
    out_df.to_csv(QC_CSV, index=False)
    print(f"[SAVED] Motion QC summary: {QC_CSV}")

# Optional: print any “bad runs” by quick heuristics:
if qc_rows:
    out_df = pd.read_csv(QC_CSV)
    flags = (
        (out_df["fd_p95"] > 0.6) |
        (out_df["n_censored"] > 20) |
        (out_df["mask_fraction"] < 0.07) |
        (out_df["mask_fraction"] > 0.40))
    bad = out_df[flags]
    if not bad.empty:
        print("\n[ATTN] Runs failing QC heuristics:")
        # 'display' is available in notebooks; if running as script, this will be ignored:
        try:
            display(bad[["subject_ID","session_ID","fd_p95","n_censored","mask_fraction","fd_plot","mask_overlay"]])
        except Exception:
            print(bad[["subject_ID","session_ID","fd_p95","n_censored","mask_fraction","fd_plot","mask_overlay"]].to_string(index=False))

    # Optional: Stage-D-only spatial QC (no atlas/GM required):
    mc_path = Path(OUT_MOTION_DIR) / f"{prefix}_bold_mc.nii"
    tsnr_median = np.nan
    support_nonzero = np.nan
    ref_mean_corr = np.nan

    if mc_path.exists():
        # Determine how many dummy frames were dropped from 'confounds.tsv':
        try:
            n_dummy = int(pd.read_csv(conf_path, sep="\t")["n_dummy_dropped"].iloc[0])
        except Exception:
            n_dummy = 0

        ref_img = nib.load(str(ref_path))
        mask_img = nib.load(str(mask_path))
        mc_img  = nib.load(str(mc_path))

        # Enforce lattice match:
        if (ref_img.shape[:3] == mc_img.shape[:3] == mask_img.shape[:3]) and \
           np.allclose(ref_img.affine, mc_img.affine, atol=1e-4) and \
           np.allclose(ref_img.affine, mask_img.affine, atol=1e-4):

            m = (mask_img.get_fdata() > 0.5)
            mc = mc_img.get_fdata(dtype=np.float32)

            # Keep post-dummy frames if they exist:
            if mc.ndim == 4 and mc.shape[3] > max(n_dummy, 1):
                mc_kept = mc[..., n_dummy:]
                # 1) Non-zero support inside mask (max across time > eps):
                eps = 1e-6
                nz = (np.max(np.abs(mc_kept), axis=3) > eps)
                support_nonzero = float(nz[m].mean()) if m.any() else np.nan

                # 2) tSNR inside mask (median over voxels):
                mu  = mc_kept.mean(axis=3)
                sig = mc_kept.std(axis=3, ddof=1)
                with np.errstate(divide='ignore', invalid='ignore'):
                    tsnr = np.where(sig > 0, mu / sig, np.nan)
                tsnr_median = float(np.nanmedian(tsnr[m])) if m.any() else np.nan

                # 3) Ref ↔ mean(mc) correlation inside mask (quick geometric/intensity sanity-check):
                ref_dat = ref_img.get_fdata(dtype=np.float32)
                mean_mc = mc_kept.mean(axis=3)
                x = ref_dat[m].ravel()
                y = mean_mc[m].ravel()
                if x.size > 10 and np.std(x) > 0 and np.std(y) > 0:
                    ref_mean_corr = float(np.corrcoef(x, y)[0, 1])
        else:
            print(f"[QC NOTE] {prefix}: skipped tSNR/nonzero support due to lattice mismatch.")

    # Include in QC row:
    qc_rows.append({
        "subject_ID": subject_ID,
        "condition_ID": condition_ID,
        "session_ID": session_ID,
        "fd_mean": fd_mean, "fd_p95": fd_p95, "fd_max": fd_max,
        "n_censored": n_cens, "n_frames_kept": n_kept,
        "mask_fraction": mask_frac, "ref_median_intensity_in_mask": med_int,
        "tsnr_median": tsnr_median,
        "support_nonzero": support_nonzero,
        "ref_mean_corr": ref_mean_corr,
        "fd_plot": str(fd_png), "mask_overlay": str(mask_png)})

--------